# SIAMA-Impute — Paso 1: Arranque, Diagnóstico y Línea Base

Este notebook parametrizado verifica:
1. La configuración centralizada (`config/settings.yaml` y `.env`) sin rutas absolutas quemadas.
2. La disponibilidad de los archivos oficiales en `data/raw/` (`estaciones.csv`, `series_diarias.csv`, `covariables.gpkg`, `test_publico.csv`), activando un fallback sintético reproducible si aún no están presentes.
3. La estructura de columnas, rango temporal, conteo de estaciones y porcentaje inicial de huecos por variable.
4. La ejecución y evaluación de la línea base inicial (**interpolación lineal con abstención y restricciones físicas** $P \ge 0$ y $T_{\max} > T_{\min}$) sin fuga de datos.

In [1]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_settings
from src.pipeline.baseline import evaluate_linear_baseline_no_leakage, run_linear_baseline
from src.pipeline.data_loader import load_climate_datasets
from src.pipeline.data_verifier import inspect_dataset

settings = load_settings(root=PROJECT_ROOT)
print(f"Proyecto: {settings.project.name} v{settings.project.version} ({settings.project.region})")
print(f"Raíz detectada: {PROJECT_ROOT}")

Proyecto: SIAMA-Impute v0.1.0 (Guanajuato, México)
Raíz detectada: E:\Hakathon SIAMA


In [2]:
# 1. Verificar archivos configurados en data/raw/
file_status = settings.check_official_files(root=PROJECT_ROOT)
for name, info in file_status.items():
    estado = "OK" if info["exists"] else "FALTA"
    print(f"[{estado:5s}] {name:18s} -> {info['path']}")

[FALTA] estaciones         -> E:\Hakathon SIAMA\data\raw\estaciones.csv
[FALTA] series_diarias     -> E:\Hakathon SIAMA\data\raw\series_diarias.csv
[FALTA] covariables        -> E:\Hakathon SIAMA\data\raw\covariables.gpkg
[FALTA] test_publico       -> E:\Hakathon SIAMA\data\raw\test_publico.csv
[OK   ] starter_notebook   -> E:\Hakathon SIAMA\notebooks\01_starter_and_baseline.ipynb


In [3]:
# 2. Cargar estaciones y series diarias (oficial o fallback sintético)
dataset = load_climate_datasets(
    settings=settings,
    root=PROJECT_ROOT,
    allow_synthetic_fallback=True,
)
print(f"Fuente de datos activa: {dataset.source}")
if dataset.missing_official_files:
    print("Archivos oficiales faltantes en data/raw/:", dataset.missing_official_files)

display(dataset.estaciones.head())
display(dataset.series_diarias.head())

Fuente de datos activa: synthetic_fallback
Archivos oficiales faltantes en data/raw/: ['estaciones', 'series_diarias', 'covariables', 'test_publico']


,id_estacion,nombre,latitud,longitud,altitud_m
0,GTO_001,León,21.122,-101.682,1815.0
1,GTO_002,Guanajuato Capital,21.019,-101.257,2000.0
2,GTO_003,Irapuato,20.676,-101.356,1730.0
3,GTO_004,Celaya,20.523,-100.815,1752.0
4,GTO_005,San Miguel de Allende,20.914,-100.744,1910.0


,id_estacion,fecha,precipitacion,tmax,tmin
0,GTO_001,2023-01-01,0.00,21.97,5.71
1,GTO_001,2023-01-02,0.00,19.26,7.89
2,GTO_001,2023-01-03,6.49,22.82,11.63
3,GTO_001,2023-01-04,0.00,23.18,9.74
4,GTO_001,2023-01-05,NaN,NaN,2.53


In [4]:
# 3. Inspeccionar columnas, rango temporal, conteo de estaciones y porcentaje de huecos
diagnostico = inspect_dataset(dataset.estaciones, dataset.series_diarias, settings=settings)
print(json.dumps(diagnostico, indent=2, ensure_ascii=False))

{
  "estaciones_count": 5,
  "estaciones_columns": [
    "id_estacion",
    "nombre",
    "latitud",
    "longitud",
    "altitud_m"
  ],
  "series_rows": 1825,
  "series_columns": [
    "id_estacion",
    "fecha",
    "precipitacion",
    "tmax",
    "tmin"
  ],
  "unique_stations_in_series": 5,
  "date_min": "2023-01-01",
  "date_max": "2023-12-31",
  "missing_pct_by_variable": {
    "precipitacion": 10.47,
    "tmax": 12.27,
    "tmin": 12.16
  },
  "physical_constraint_violations": {
    "precip_negative_count": 0,
    "tmax_le_tmin_count": 0
  }
}


In [5]:
# 4. Evaluar línea base (interpolación lineal) sin fuga de datos
metricas_baseline = evaluate_linear_baseline_no_leakage(
    dataset.series_diarias,
    settings=settings,
)
print(json.dumps(metricas_baseline, indent=2, ensure_ascii=False))

{
  "method": "linear_interpolation",
  "max_gap_days": 7,
  "validation_mask_fraction": 0.1,
  "random_seed": 42,
  "metrics_by_variable": {
    "precipitacion": {
      "n_eval": 163,
      "n_imputed": 161,
      "n_abstained": 2,
      "coverage_pct": 98.77,
      "mae": 2.298,
      "rmse": 4.763
    },
    "tmax": {
      "n_eval": 160,
      "n_imputed": 158,
      "n_abstained": 2,
      "coverage_pct": 98.75,
      "mae": 2.0045,
      "rmse": 2.4897
    },
    "tmin": {
      "n_eval": 160,
      "n_imputed": 159,
      "n_abstained": 1,
      "coverage_pct": 99.38,
      "mae": 2.859,
      "rmse": 3.5419
    }
  }
}


In [6]:
# 5. Ejecutar imputación baseline con banderas explícitas (observado / imputado_baseline / abstenido)
series_baseline = run_linear_baseline(dataset.series_diarias, settings=settings)
for var in settings.data_schema.variables:
    flag_col = f"{var}_flag"
    if flag_col in series_baseline.columns:
        print(f"\nDistribución de banderas para {var}:")
        print(series_baseline[flag_col].value_counts(dropna=False))

# Guardar resumen reproducible en data/processed/
out_dir = settings.paths.processed_path(root=PROJECT_ROOT)
out_dir.mkdir(parents=True, exist_ok=True)
report_path = out_dir / "notebook_baseline_metrics.json"
with report_path.open("w", encoding="utf-8") as fh:
    json.dump(
        {
            "source": dataset.source,
            "diagnostico": diagnostico,
            "metricas_baseline": metricas_baseline,
        },
        fh,
        indent=2,
        ensure_ascii=False,
    )
print(f"\nResumen guardado en: {report_path}")


Distribución de banderas para precipitacion:
precipitacion_flag
observado            1634
imputado_baseline     188
abstenido               3
Name: count, dtype: int64

Distribución de banderas para tmax:
tmax_flag
observado            1601
imputado_baseline     223
abstenido               1
Name: count, dtype: int64

Distribución de banderas para tmin:
tmin_flag
observado            1603
imputado_baseline     221
abstenido               1
Name: count, dtype: int64

Resumen guardado en: E:\Hakathon SIAMA\data\processed\notebook_baseline_metrics.json
